# Parsing object dependency using NetworkX

In [1]:
# this magic for develop only
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [3]:
import networkx as nx
import polars as pl
import xml.etree.ElementTree as ET
import xmlschema

In [4]:
from ggblab import GeoGebra
# from ggblab.parser import tokenize_with_commas

In [79]:
# initialize base class, not open GeoGebra Widget
ggb = GeoGebra()

Using local cached file: xsd/common.xsd


In [86]:
# import sys
# from pathlib import Path
# sys.path.insert(0, str(Path.cwd().parent))

from ggblab_extra.dataframe_io import DataFrameIO

In [81]:
await ggb.init()

In [ ]:
c = ggb.construction.load('2025_13_01.ggb')

In [103]:
import xml.etree.ElementTree as ET
root = ET.Element(c.geogebra_xml)
tree = ET.ElementTree(root)
tree.write("2025_13_01.xml", encoding='utf-8', xml_declaration=True)

In [82]:
r = await ggb.function("setBase64", [ggb.construction.base64_buffer.decode('utf-8')])

In [93]:
p.df = await DataFrameIO.initialize_dataframe(ggb, file='2025_13_01.ggb', _columns=p.COLUMNS + ["ShowObject", "ShowLabel", "Auxiliary"])
p.df2 = await DataFrameIO.initialize_dataframe(ggb, use_applet=True)

In [14]:
from ggblab_extra import ggb_parser

In [92]:
p.COLUMNS + ["ShowObject", "ShowLabel", "Auxiliary"]

['Type',
 'Command',
 'Value',
 'Caption',
 'Layer',
 'ShowObject',
 'ShowLabel',
 'Auxiliary']

In [66]:
r = await ggb.function("getAllObjectNames")
r

['C',
 'A',
 'poly1',
 'f',
 'g',
 'E',
 'D',
 'h',
 'i',
 'O',
 'c_1',
 'P',
 'j',
 'k',
 'l',
 'm',
 'n',
 'p',
 'q',
 'r',
 'poly2',
 's',
 't',
 'G',
 'H',
 'a_{3}',
 'b_1',
 'e',
 "P'",
 'd',
 'B',
 'f_1',
 'g_{4}',
 'w',
 'c_{5}',
 'I',
 'J',
 'K',
 'u',
 'v',
 'h_1',
 'L',
 'proj_{u}w',
 'proj_{v}w',
 'γ',
 'β',
 'α',
 'θ',
 'b',
 'c',
 "Γ'_{uu}",
 'Γ_{uu}',
 'u_1',
 'v_1',
 'δ',
 'ε',
 'text1',
 'text3',
 'text3_{1}',
 'text5',
 'i_1',
 't1',
 'p_1',
 'o',
 'c_3',
 't2',
 'p_2',
 'o_1',
 'a_1',
 't3',
 'p_3',
 'a_2',
 'c_4',
 'poly3',
 'j_1',
 'k_1',
 'F',
 'M',
 'l_1',
 'm_1',
 'poly4',
 'n_1',
 'q_1',
 'N',
 'Q',
 'r_1',
 's_1',
 't_1',
 'd_1',
 'e_1',
 'f_2',
 'g_{5}',
 'h_2',
 'a',
 'poly5',
 'i_2',
 'j_2',
 'R',
 'S',
 'k_2',
 'l_2',
 'poly6',
 'm_2',
 'n_2',
 'T',
 'U',
 'q_2',
 'r_2',
 "w'",
 't4',
 'c_2',
 'a_4',
 'b_2',
 'text4',
 's_2',
 't_2',
 'V',
 'poly7',
 'd_2',
 'e_2',
 'W',
 'Z',
 'f_3',
 'g_{6}',
 'text3_{2}',
 'text5_{1}',
 'h_3',
 "O'",
 'g_2',
 'g_1',
 't5

In [71]:
set(p.df2["Type"].unique()) - set(p.df["Type"].unique())

{'circle', 'quadrilateral', 'triangle'}

In [72]:
set(p.df["Type"].unique()) - set(p.df2["Type"].unique())

{'conic'}

In [98]:
import os
os.path.splitext(ggb.file.source_file)[0]+'.json'

'2025_13_01.json'

In [100]:
p.df.write_json(os.path.splitext(ggb.file.source_file)[0]+'.json')

In [62]:
# p.df2["Command"] == p.df["Command"]
mask = p.df['Command'].eq_missing(p.df2['Command']).not_()
p.df2.filter(mask)

Name,Type,Command,Value,Caption,Layer
str,str,str,str,str,i64
"""E""","""point""","""Polygon(C, A, 4)""","""E = (2.1, 2.2)""",null,2
"""D""","""point""","""Polygon(C, A, 4)""","""D = (0, 2.2)""",null,2
"""h""","""segment""","""Segment(E, D, poly1)""","""h = 2.2""",null,2
"""i""","""segment""","""Segment(D, C, poly1)""","""i = 2.2""",null,2
"""G""","""point""","""Polygon(A, C, 4)""","""G = (0, -2.2)""",null,4
"""H""","""point""","""Polygon(A, C, 4)""","""H = (2.2, -2.1)""",null,4
"""a_{3}""","""segment""","""Segment(G, H, poly2)""","""a_{3} = 2.2""",null,4
"""b_1""","""segment""","""Segment(H, A, poly2)""","""b_1 = 2.2""",null,4
"""proj_{u}w""","""numeric""","""(w u) / (u u)""","""proj_{u}w = 1.2""",null,8


In [63]:
p.df.filter(mask)

Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary
str,str,str,str,str,i64,bool,bool,bool
"""h""","""segment""","""Segment(E, D, poly1)""",null,null,2,false,false,true
"""i""","""segment""","""Segment(D, C, poly1)""",null,null,2,false,false,true
"""E""","""point""","""Polygon(C, A, 4)""",null,null,2,true,false,true
"""D""","""point""","""Polygon(C, A, 4)""",null,null,2,true,false,true
"""a_{3}""","""segment""","""Segment(G, H, poly2)""",null,null,4,false,false,true
"""b_1""","""segment""","""Segment(H, A, poly2)""",null,null,4,false,false,true
"""G""","""point""","""Polygon(A, C, 4)""",null,null,4,true,false,true
"""H""","""point""","""Polygon(A, C, 4)""",null,null,4,true,false,true
"""proj_{u}w""","""numeric""","""((w * u)) / ((u * u))""","""((w * u)) / ((u * u))""",null,0,false,false,false


In [7]:
# load from .ggb (zipped, and base64 encoded)
c = ggb.construction.load('2025_13_01.ggb')
# c = ggb.construction.load('2025_06_08.ggb')
# c = ggb.construction.load('2025_15_01.ggb')

In [22]:
# parse loaded xml as python dict
o = c.ggb_schema.decode(c.geogebra_xml)

In [23]:
# check command part in archive file, and found not enough information for construction...
for _c in o["command"]:
    _ci = tuple(zip(*_c['input'].items()))[1]
    _co = tuple(zip(*_c['output'].items()))[1]
    print(f"{', '.join(_co)} = {_c['@name']}[{', '.join(_ci)}]")

poly1, f, g, h, i, E, D = Polygon[C, A, 4]
O = Midpoint[C, A]
c_1 = Circle[O, C]
P = Point[c_1]
j = Ray[C, P]
k = Ray[A, P]
l = OrthogonalLine[E, k]
m = OrthogonalLine[D, j]
n = Line[D, j]
p = Line[E, m]
q = Line[A, l]
r = Line[C, k]
poly2, s, t, a_{3}, b_1, G, H = Polygon[A, C, 4]
e = Ray[C, A]
P' = Point[c_1]
d = Ray[A, P']
B = Intersect[d, j]
f_1 = Line[C, d]
g_{4} = OrthogonalLine[C, d]
w = Vector[C, B]
c_{5} = Circle[C, 1]
I,  = Intersect[c_{5}, e]
J, K = Intersect[c_{5}, f_1]
u = Vector[C, I]
v = Vector[C, K]
h_1 = OrthogonalLine[B, e]
L = Intersect[h_1, e]
γ = Angle[A, C, B]
β = Angle[A, B, C]
α = Angle[C, A, B]
θ = Angle[Vector[C, A], Vector[A, B]]
b = Distance[C, A]
c = Distance[A, B]
u_1 = Translate[Vector[(((u * w)) / ((u * u)) * u)], C]
v_1 = Translate[Vector[(((w * v)) / ((v * v)) * v)], P']
δ = Angle[C, P, A]
ε = Angle[C, P', A]
i_1 = Segment[P, O]
t1, p_1, o, c_3 = Polygon[O, C, P]
t2, p_2, o_1, a_1 = Polygon[O, A, P]
t3, p_3, a_2, c_4 = Polygon[A, C, P]
poly3, j_1, k_1,

## with Applet

In [10]:
r = await ggb.init()
r

In [11]:
# ggb.construction.base64_buffer.decode('utf-8')

In [12]:
r = await ggb.function("setBase64", [ggb.construction.base64_buffer.decode('utf-8')])

In [13]:
ggb.comm.target_comm

In [14]:
# compare in archive and in applet
for _c in o["command"]:
    _ci = tuple(zip(*_c['input'].items()))[1]
    _co = tuple(zip(*_c['output'].items()))[1]
    c1 = (f"{_c['@name']}({', '.join(_ci)})"
          .replace('OrthogonalLine', 'PerpendicularLine')
          .translate(str.maketrans('[]', '()')))
    c2 = await ggb.function("getCommandString", [_c['output']['@a0']])
    if (c1 != c2):
        # print(f"{', '.join(_co)} = {c1}")
        print(f"{_c['output']['@a0']}: {c1}, {c2}")

u_1: Translate(Vector((((u * w)) / ((u * u)) * u)), C), Translate((u w) / (u u) u, C)
v_1: Translate(Vector((((w * v)) / ((v * v)) * v)), P'), Translate((w v) / (v v) v, P')
w': Translate(Vector((Γ_{uu} * u) + (Γ'_{uu} * v)), C), Translate(Γ_{uu} u + Γ'_{uu} v, C)


In [15]:
for _e in o["expression"]:
    c1 = (_e['@exp']
          .translate(str.maketrans('[]', '()')))
    c2 = await ggb.function("getCommandString", [_e['@label']])
    if (c1 != c2):
        print(_e['@label'], c1, c2)

proj_{u}w ((w * u)) / ((u * u)) (w u) / (u u)
proj_{v}w ((w * v)) / ((v * v)) (w v) / (v v)
Γ'_{uu} (-(((proj_{u}w * cos(θ)) - proj_{v}w) / (sin(θ))^(2))) -((proj_{u}w cos(θ) - proj_{v}w) / sin²(θ))
Γ_{uu} (proj_{u}w - (proj_{v}w * cos(θ))) / (sin(θ))^(2) (proj_{u}w - proj_{v}w cos(θ)) / sin²(θ)
text1 "1.  Thales's  theorem: right  triangle  inscribed  in  a  circle" None
text3 "3.  Pythagorean  theorem" None
text3_{1} "$\|a\| = \|b\| \cos(\gamma)+ \|c\| \cos(\beta)$
$\|b\| = \|c\| \cos(\alpha) + \|a\| \cos(\gamma)$
$\|c\| = \|a\| \cos(\beta) + \|b\| \cos(\alpha)$" None
text5 "5.  Projection" None
text4 "4.  Orthocenter  and  Law  of  cosines" None
text3_{2} "$\|a\|^2 = \|b\|^2 + \|c\|^2 - 2 \|b\|\|c\| \cos(\alpha)$
$\|b\|^2 = \|c\|^2 + \|a\|^2 - 2 \|c\|\|a\| \cos(\beta)$
$\|c\|^2 = \|a\|^2 + \|b\|^2 - 2 \|a\|\|b\| \cos(\gamma)$" None
text5_{1} "$\|b\| \cos(\theta) = \frac{\vec{b}\cdot\vec{a}}{\vec{a}\cdot\vec{a}}$" None
g_3 ((Area(t5) * g_2) * g_2) Area(t5) g_2 g_2
text2 "2.  Geometri

In [160]:
# construction protocol as list
def dummy():
    for e in o['element']:
        _n = e['@label']
        cmd = None
        exp = None
        for _c in o['command']:
            _ci = tuple(zip(*_c['input'].items()))[1]
            _co = tuple(zip(*_c['output'].items()))[1]
            def build_command_string(edges, vertices):
                nonlocal _n
                nonlocal _ci, _co
                nonlocal cmd, ci , co
                try:
                    _i = edges.index(_n)
                    ci = (vertices[_i:] + vertices[:_i])[:2] + (_co[0],)
                    co = _co[0]
                    cmd = f"Segment({', '.join(ci)})"
                except:
                    pass
            if _n in _co:
                # print(_c['@name'], _ci, _co)
                if _c['@name'] == 'Polygon':
                    match _ci:
                        case (p0, p1, '4'):
                            _, e0, e1, e2, e4, p2, p3 = _co
                            edges = (e0, e1, e2, e4)
                            vertices = (p0, p1, p2, p3)
                            build_command_string(edges, vertices)
                            break
                        case _:
                            edges = _co[1:]
                            vertices = _ci
                            build_command_string(edges, vertices)             
                            break
                ci = _ci
                co = _co
                cmd = (f"{_c['@name']}({', '.join(_ci)})"
                      .replace('OrthogonalLine', 'PerpendicularLine')
                      .translate(str.maketrans('[]', '()')))
                break
        for _e in o['expression']:
            if _n == _e['@label']:
                exp = _e['@exp']
        
        # if (e['@type'] != 'text') and (cmd or exp):
        #     r = await ggb.function("getCommandString", [_n])
        #     if r != (cmd or exp):
        #         print(_n, e['@type'], cmd or exp, r)
        # r = await ggb.function("getValueString", [_n])
    
        print([_n, e['@type'], cmd or exp, 
               e.get('caption', [{}])[0].get('@val'),
               e.get('layer', [{}])[0].get('@val'),
               e.get('show', [{}])[0].get('@object'), 
               e.get('show', [{}])[0].get('@label'), 
               e.get('auxiliary', [{}])[0].get('@val'), 
              ])

dummy()

['C', 'point', None, None, 9, True, True, None]
['A', 'point', None, None, 9, True, True, None]
['poly1', 'polygon', None, None, 3, False, False, True]
['f', 'segment', 'Segment(C, A, poly1)', None, 2, False, False, True]
['g', 'segment', 'Segment(A, E, poly1)', None, 2, False, False, True]
['h', 'segment', 'Segment(E, D, poly1)', None, 2, False, False, True]
['i', 'segment', 'Segment(D, C, poly1)', None, 2, False, False, True]
['E', 'point', None, None, 2, True, False, True]
['D', 'point', None, None, 2, True, False, True]
['O', 'point', 'Midpoint(C, A)', None, 1, False, True, None]
['c_1', 'conic', 'Circle(O, C)', None, 0, True, False, True]
['P', 'point', 'Point(c_1)', None, 9, True, True, None]
['j', 'ray', 'Ray(C, P)', None, 0, True, False, None]
['k', 'ray', 'Ray(A, P)', None, 0, True, False, None]
['l', 'line', 'PerpendicularLine(E, k)', None, 3, False, False, True]
['m', 'line', 'PerpendicularLine(D, j)', None, 3, False, False, True]
['n', 'line', 'Line(D, j)', None, 3, False, 

## Construction Protocol

In [73]:
ggb.parser.parse()

AttributeError: 'ggb_parser' object has no attribute 'parse'

In [21]:
ggb.parser.command_cache.get_all()

{'Angle': 8,
 'Area': 1,
 'Circle': 59,
 'Distance': 5,
 'Intersect': 110,
 'Line': 109,
 'Midpoint': 21,
 'PerpendicularLine': 98,
 'Point': 42,
 'Polygon': 153,
 'Ray': 84,
 'Segment': 229,
 'Translate': 3,
 'Vector': 64,
 'cos': 2,
 'sin²': 2}

In [74]:
p.parse()

In [75]:
nx.write_network_text(p.G)

╟── C
╎   ├─╼ poly1 ╾ A
╎   │   ├─╼ f ╾ C, A
╎   │   ├─╼ g ╾ A, E
╎   │   ├─╼ h ╾ E, D
╎   │   └─╼ i ╾ D, C
╎   ├─╼ E ╾ A
╎   │   ├─╼ l ╾ k
╎   │   │   └─╼ q ╾ A
╎   │   ├─╼ p ╾ m
╎   │   │   └─╼ V ╾ n
╎   │   │       ├─╼ poly7 ╾ M
╎   │   │       │   ├─╼ d_2 ╾ V, M
╎   │   │       │   ├─╼ e_2 ╾ M, W
╎   │   │       │   ├─╼ f_3 ╾ W, Z
╎   │   │       │   └─╼ g_{6} ╾ Z, V
╎   │   │       ├─╼ W ╾ M
╎   │   │       │   └─╼  ...
╎   │   │       ├─╼ Z ╾ M
╎   │   │       │   └─╼  ...
╎   │   │       └─╼  ...
╎   │   └─╼  ...
╎   ├─╼ D ╾ A
╎   │   ├─╼ m ╾ j
╎   │   │   └─╼  ...
╎   │   ├─╼ n ╾ j
╎   │   │   └─╼  ...
╎   │   └─╼  ...
╎   ├─╼ O ╾ A
╎   │   ├─╼ c_1 ╾ C
╎   │   │   ├─╼ P
╎   │   │   │   ├─╼ j ╾ C
╎   │   │   │   │   ├─╼ B ╾ d
╎   │   │   │   │   │   ├─╼ w ╾ C
╎   │   │   │   │   │   │   ├─╼ u_1 ╾ u, C
╎   │   │   │   │   │   │   └─╼ v_1 ╾ v, P'
╎   │   │   │   │   │   ├─╼ h_1 ╾ e
╎   │   │   │   │   │   │   └─╼ L ╾ e
╎   │   │   │   │   │   │       ├─╼ t_1 ╾ C
╎   │   │   │   │ 

In [76]:
p.parse_subgraph()

found: 'C' => 'c_{5}'
found: 'C', 'A' => 'e'
found: 'C', 'A' => 'E'
found: 'C', 'A' => 'D'
found: 'C', 'A' => 'O'
found: 'O' => 'c_1'
found: 'c_{5}', 'e' => 'I'
found: 'c_1' => 'P''
found: 'c_1' => 'P'
found: 'P'' => 'd'
found: 'P' => 'j'
found: 'P' => 'F'
found: 'P' => 'k'
found: 'P' => 'h_3'
found: 'P' => 'M'
found: 'j' => 'm'
found: 'j' => 'n'
found: 'h_3' => 'O''
found: 'd' => 'f_1'
found: 'd', 'j' => 'B'
found: 'f_1' => 'K'
found: 'O'' => 'I_1'
found: 'O'' => 'd_3'
found: 'B' => 'h_1'
found: 'm' => 'p'
found: 'd_3' => 'A_1'
found: 'p' => 'V'
found: 'A_1' => 'k_3'
found: 'A_1' => 'l_3'
found: 'k_3' => 'r_3'
found: 'l_3' => 'B_1'
found: 'r_3' => 'C_1'


In [77]:
nx.write_network_text(p.G2)

╟── C
╎   ├─╼ c_{5}
╎   │   └─╼ I ╾ e
╎   ├─╼ e ╾ A
╎   │   └─╼  ...
╎   ├─╼ E ╾ A
╎   ├─╼ D ╾ A
╎   └─╼ O ╾ A
╎       └─╼ c_1
╎           ├─╼ P'
╎           │   └─╼ d
╎           │       ├─╼ f_1
╎           │       │   └─╼ K
╎           │       └─╼ B ╾ j
╎           │           └─╼ h_1
╎           └─╼ P
╎               ├─╼ j
╎               │   ├─╼ m
╎               │   │   └─╼ p
╎               │   │       └─╼ V
╎               │   ├─╼ n
╎               │   └─╼  ...
╎               ├─╼ F
╎               ├─╼ k
╎               ├─╼ h_3
╎               │   └─╼ O'
╎               │       ├─╼ I_1
╎               │       └─╼ d_3
╎               │           └─╼ A_1
╎               │               ├─╼ k_3
╎               │               │   └─╼ r_3
╎               │               │       └─╼ C_1
╎               │               └─╼ l_3
╎               │                   └─╼ B_1
╎               └─╼ M
╙── A
    └─╼  ...


In [24]:
labels_map = {}
for n in p.ft:
    if n in p.G2:
        labels_map[n] = f"[{n}: {len(nx.descendants(p.G, n))}]"

In [25]:
nx.set_node_attributes(p.G, labels_map, "label")
nx.write_network_text(p.G, with_labels="label")

╟── [C: 141]
╎   ├─╼ poly1 ╾ [A: 140]
╎   │   ├─╼ f ╾ [C: 141], [A: 140]
╎   │   ├─╼ g ╾ [A: 140], [E: 13]
╎   │   ├─╼ h ╾ [E: 13], [D: 13]
╎   │   └─╼ i ╾ [D: 13], [C: 141]
╎   ├─╼ [E: 13] ╾ [A: 140]
╎   │   ├─╼ l ╾ [k: 3]
╎   │   │   └─╼ q ╾ [A: 140]
╎   │   ├─╼ [p: 8] ╾ [m: 9]
╎   │   │   └─╼ [V: 7] ╾ [n: 8]
╎   │   │       ├─╼ poly7 ╾ [M: 9]
╎   │   │       │   ├─╼ d_2 ╾ [V: 7], [M: 9]
╎   │   │       │   ├─╼ e_2 ╾ [M: 9], W
╎   │   │       │   ├─╼ f_3 ╾ W, Z
╎   │   │       │   └─╼ g_{6} ╾ Z, [V: 7]
╎   │   │       ├─╼ W ╾ [M: 9]
╎   │   │       │   └─╼  ...
╎   │   │       ├─╼ Z ╾ [M: 9]
╎   │   │       │   └─╼  ...
╎   │   │       └─╼  ...
╎   │   └─╼  ...
╎   ├─╼ [D: 13] ╾ [A: 140]
╎   │   ├─╼ [m: 9] ╾ [j: 39]
╎   │   │   └─╼  ...
╎   │   ├─╼ [n: 8] ╾ [j: 39]
╎   │   │   └─╼  ...
╎   │   └─╼  ...
╎   ├─╼ [O: 122] ╾ [A: 140]
╎   │   ├─╼ [c_1: 119] ╾ [C: 141]
╎   │   │   ├─╼ [P: 109]
╎   │   │   │   ├─╼ [j: 39] ╾ [C: 141]
╎   │   │   │   │   ├─╼ [B: 27] ╾ [d: 34]
╎   │   │   │   